In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 22.6 MB/s eta 0:00:00


In [4]:
import warnings
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import KFold
from sklearn.base import clone
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error
from imblearn.over_sampling import SMOTE
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
import joblib
from scipy.stats import randint, uniform
from sklearn.model_selection import PredefinedSplit

warnings.filterwarnings("ignore", category=FutureWarning)

# -------------------- Constants & Helper Functions -------------------- #
TEAM_MAPPING = {
    'Man Utd': 'Manchester Utd', 'Man United': 'Manchester Utd',
    'Man City': 'Manchester City', 'Newcastle Utd': 'Newcastle United',
    'Newcastle Ut': 'Newcastle United', "Nott'ham Forest": 'Nottingham Forest',
    'Paris S-G': 'Paris Saint-Germain', 'Inter Milan': 'Inter',
    'Spurs': 'Tottenham', 'West Ham Utd': 'West Ham United'
}

def clean_numeric_values(value):
    if isinstance(value, str):
        cleaned = value.replace(',', '').replace(' ', '')
        if cleaned.replace('.', '', 1).isdigit():
            return float(cleaned)
    return value

def standardize_team_names(df, column_name):
    df = df.copy()
    df.loc[:, column_name] = df[column_name].replace(TEAM_MAPPING).str.strip()
    return df

# -------------------- Custom Group K-Fold Splitter -------------------- #
class TeamBasedGroupKFold:
    def __init__(self, n_splits=5, random_state=None):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, fixtures_df):
        unique_teams = pd.concat([fixtures_df['Home_Team'], fixtures_df['Away_Team']]).unique()
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)

        for _, test_teams_idx in kf.split(unique_teams):
            test_teams = unique_teams[test_teams_idx]
            val_mask = fixtures_df['Home_Team'].isin(test_teams) | fixtures_df['Away_Team'].isin(test_teams)
            train_idx = fixtures_df.index[~val_mask].values
            val_idx = fixtures_df.index[val_mask].values
            yield train_idx, val_idx

# -------------------- Data Loading & Preprocessing -------------------- #
def load_and_preprocess_data():
    stats_df = pd.read_csv('/content/Combined_Leagues_Stats.csv').copy()
    fixtures_df = pd.read_csv('/content/Fixture_Results.csv').copy()

    numeric_cols = [
        'progressive_carries', 'progressive_passes', 'xg', 'npxg',
        'xg_assist', 'npxg_xg_assist', 'goals_per90', 'assists_per90',
        'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
        'xg_assist_per90', 'npxg_per90', 'Home_xG', 'Away_xG'
    ]

    for df in [stats_df, fixtures_df]:
        for col in numeric_cols:
            if col in df.columns:
                df.loc[:, col] = df[col].apply(clean_numeric_values)

    # Reset index after dropping rows
    fixtures_df = fixtures_df.dropna(subset=['Home_xG', 'Away_xG']).copy()
    fixtures_df.reset_index(drop=True, inplace=True)  # <-- FIX 1

    stats_df = standardize_team_names(stats_df, 'team')
    fixtures_df = standardize_team_names(fixtures_df, 'Home_Team')
    fixtures_df = standardize_team_names(fixtures_df, 'Away_Team')

    # Reset index after filtering
    fixtures_df = fixtures_df.loc[fixtures_df['Home_Team'] != fixtures_df['Away_Team']].copy()
    fixtures_df.reset_index(drop=True, inplace=True)  # <-- FIX 2

    stats_df.loc[:, 'total_progression'] = (
        stats_df['progressive_carries'] + stats_df['progressive_passes']
    )

    feature_columns = [
        'total_progression', 'goals_per90', 'assists_per90',
        'goals_assists_per90', 'goals_pens_per90', 'xg_per90',
        'xg_assist_per90', 'npxg_per90'
    ]

    imputer = SimpleImputer(strategy='median')
    stats_df.loc[:, feature_columns] = imputer.fit_transform(stats_df[feature_columns])

    team_stats = stats_df.set_index('team')[feature_columns].to_dict('index')

    return team_stats, fixtures_df, feature_columns

# -------------------- TeamBasedGroupKFold Splitter -------------------- #
class TeamBasedGroupKFold:
    def __init__(self, n_splits=5, random_state=None):
        self.n_splits = n_splits
        self.random_state = random_state

    def split(self, fixtures_df):
        unique_teams = pd.concat([fixtures_df['Home_Team'], fixtures_df['Away_Team']]).unique()
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)

        for _, test_teams_idx in kf.split(unique_teams):
            test_teams = unique_teams[test_teams_idx]
            val_mask = (
                fixtures_df['Home_Team'].isin(test_teams) |
                fixtures_df['Away_Team'].isin(test_teams)
            )
            # Convert boolean mask to positional indices
            val_idx = val_mask.to_numpy().nonzero()[0]  # <-- FIX 3
            train_idx = (~val_mask).to_numpy().nonzero()[0]
            yield train_idx, val_idx
# -------------------- Optuna Objective Function -------------------- #
def create_objective(X_train, y_class_train, y_reg_train):
    def objective(trial):
        params = {
            'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0, 0.5),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0),
            'objective': 'multi:softprob',
            'num_class': 3,
            'random_state': 42,
            'eval_metric': 'mlogloss',
            'early_stopping_rounds': 20  # Moved to constructor parameters
        }

        # Handle class imbalance
        class_weights = compute_class_weight('balanced', classes=np.unique(y_class_train), y=y_class_train)

        # Create pipeline with SMOTE
        pipeline = ImbPipeline([
            ('smote', SMOTE(random_state=42)),
            ('clf', xgb.XGBClassifier(**params))
        ])

        # Inner cross-validation
        scores = []
        inner_cv = TeamBasedGroupKFold(n_splits=3, random_state=42)
        for inner_train_idx, inner_val_idx in inner_cv.split(fixtures_df.iloc[train_idx]):
            X_inner_train = X_train[inner_train_idx]
            y_inner_train = y_class_train[inner_train_idx]

            # Apply SMOTE and sample weights
            X_res, y_res = SMOTE(random_state=42).fit_resample(X_inner_train, y_inner_train)
            sample_weights_res = class_weights[y_res]

            # Modified fit call
            pipeline.named_steps['clf'].fit(
                X_res, y_res,
                sample_weight=sample_weights_res,
                eval_set=[(X_train[inner_val_idx], y_class_train[inner_val_idx])],
                verbose=False
            )

            y_pred = pipeline.predict(X_train[inner_val_idx])
            scores.append(f1_score(y_class_train[inner_val_idx], y_pred, average='macro'))

        return np.mean(scores)
    return objective

# -------------------- Training Execution -------------------- #
if __name__ == "__main__":
    team_stats, fixtures_df, feature_columns = load_and_preprocess_data()

    # Feature engineering
    fixtures_df['features'] = fixtures_df.apply(
        lambda row: (
            list(team_stats[row['Home_Team'].strip()].values()) +  # Fixed closing )
            list(team_stats[row['Away_Team'].strip()].values())    # Fixed closing )
        ), axis=1
    )

    fixtures_df['result'] = fixtures_df.apply(
        lambda row: 2 if row['Home_Score'] > row['Away_Score'] else 1 if row['Home_Score'] == row['Away_Score'] else 0,
        axis=1
    )

    X = np.array(fixtures_df['features'].tolist())
    y_class = fixtures_df['result'].values
    y_reg = fixtures_df[['Home_xG', 'Away_xG']].values

    group_kfold = TeamBasedGroupKFold(n_splits=5, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(fixtures_df)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_class_train, y_class_val = y_class[train_idx], y_class[val_idx]
        y_reg_train, y_reg_val = y_reg[train_idx], y_reg[val_idx]

        # Corrected Optuna optimization call
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(
            create_objective(X_train, y_class_train, y_reg_train),
            n_trials=50
        )

        best_params = study.best_params

        # Now use best_params
        # Instantiate classifier with early stopping and eval_metric in the constructor
        clf = xgb.XGBClassifier(
            **best_params,
            early_stopping_rounds=20,
            eval_metric="mlogloss"
        )

        # ======== CRITICAL FIX ========
        # 1. Apply SMOTE to training data
        smote = SMOTE(random_state=42)
        X_res, y_res = smote.fit_resample(X_train, y_class_train)
        # ==============================

        # 2. Train classifier with early stopping (parameters passed in constructor)
        clf.fit(
            X_res, y_res,
            eval_set=[(X_val, y_class_val)],
            # early_stopping_rounds=20, # REMOVE - passed in constructor
            verbose=False
            # eval_metric="mlogloss" # REMOVE - passed in constructor
        )

        # 3. Create a clean classifier copy without early stopping for calibration
        # Early stopping should not be used during calibration fit
        calib_clf_params = {k: v for k, v in best_params.items() if k not in ['early_stopping_rounds', 'eval_metric']}
        calib_clf = xgb.XGBClassifier(**calib_clf_params)
        calib_clf.fit(X_res, y_res) # Train on SMOTE data without early stopping parameters

        # 4. Calibrate using original training splits
        calibrated_clf = CalibratedClassifierCV(
            calib_clf,
            method='sigmoid',
            cv=KFold(n_splits=5, shuffle=True, random_state=42)
        )
        calibrated_clf.fit(X_train, y_class_train)

        # 5. Train regression models - Use best_params, early stopping and eval_metric usually not needed for regressor fit unless you specifically want it.
        reg_home_params = {k: v for k, v in best_params.items() if k not in ['objective', 'num_class', 'eval_metric', 'early_stopping_rounds']}
        reg_away_params = {k: v for k, v in best_params.items() if k not in ['objective', 'num_class', 'eval_metric', 'early_stopping_rounds']}

        # Note: Resampling y_reg_train for regression needs care.
        # A simple modulo can work if the resampling preserves order, but it's potentially fragile.
        # A safer approach might be to use the original y_reg_train and handle sample weights if needed,
        # or ensure the resampling method handles multi-output targets correctly.
        # For now, keeping the original logic but noting this is a potential area for refinement.
        y_reg_train_res = y_reg_train[np.arange(len(X_res)) % len(y_reg_train)]

        reg_home = xgb.XGBRegressor(**reg_home_params).fit(X_res, y_reg_train_res[:, 0])
        reg_away = xgb.XGBRegressor(**reg_away_params).fit(X_res, y_reg_train_res[:, 1])

        # Save model artifacts
        joblib.dump({
            'classifier': calibrated_clf,
            'regressor_home': reg_home,
            'regressor_away': reg_away,
            'feature_columns': feature_columns,
            'team_stats': team_stats,
            'best_params': best_params
        }, f'xgb_model_fold{fold+1}.pkl')

        # Validation metrics
        print(f"\nFold {fold+1} Metrics:")
        print(f"Val Accuracy: {accuracy_score(y_class_val, calibrated_clf.predict(X_val)):.4f}")
        print(f"Val F1: {f1_score(y_class_val, calibrated_clf.predict(X_val), average='macro'):.4f}")

    joblib.dump({
        'team_stats': team_stats,
        'feature_columns': feature_columns
    }, 'xgb_artifacts.pkl')

# -------------------- Ensemble Prediction Function -------------------- #
def predict_fixtures(fixtures_csv_path):
    artifacts = joblib.load('xgb_artifacts.pkl')
    team_stats = artifacts['team_stats']
    feature_columns = artifacts['feature_columns']

    # Load all fold models
    models = [joblib.load(f'xgb_model_fold{i+1}.pkl') for i in range(5)]

    # Prepare new fixtures
    new_fixtures = pd.read_csv(fixtures_csv_path)
    new_fixtures = standardize_team_names(new_fixtures, 'Home_Team')
    new_fixtures = standardize_team_names(new_fixtures, 'Away_Team')

    new_fixtures['features'] = new_fixtures.apply(
            lambda row: (
                list(team_stats[row['Home_Team'].strip()].values()) +  # Fixed closing )
                list(team_stats[row['Away_Team'].strip()].values())    # Fixed closing )
            ), axis=1
        )
    X_new = np.array(new_fixtures['features'].tolist())

    # Ensemble predictions
    class_probs = np.zeros((X_new.shape[0], 3))
    home_xg = np.zeros(X_new.shape[0])
    away_xg = np.zeros(X_new.shape[0])

    for model in models:
        class_probs += model['classifier'].predict_proba(X_new)
        home_xg += model['regressor_home'].predict(X_new)
        away_xg += model['regressor_away'].predict(X_new)

    return pd.DataFrame({
        'Home_Team': new_fixtures['Home_Team'],
        'Away_Team': new_fixtures['Away_Team'],
        'Home_Win_Prob': class_probs[:, 2] / len(models),
        'Draw_Prob': class_probs[:, 1] / len(models),
        'Away_Win_Prob': class_probs[:, 0] / len(models),
        'Predicted_Home_xG': home_xg / len(models),
        'Predicted_Away_xG': away_xg / len(models)
    })

# Example usage:
# predictions = predict_fixtures('fixtures.csv')
# predictions['Prob_Diff'] = abs(predictions['Home_Win_Prob'] - predictions['Away_Win_Prob'])
# final_predictions = predictions.sort_values('Prob_Diff', ascending=False)

[I 2025-05-16 12:12:20,801] A new study created in memory with name: no-name-63f7d4ed-1f8c-4528-aaef-3525b270d79c
[I 2025-05-16 12:12:22,387] Trial 0 finished with value: 0.41395181451450397 and parameters: {'booster': 'dart', 'learning_rate': 0.1205712628744377, 'n_estimators': 340, 'max_depth': 4, 'min_child_weight': 2, 'gamma': 0.02904180608409973, 'subsample': 0.9330880728874675, 'colsample_bytree': 0.8005575058716043, 'reg_alpha': 0.7080725807153198, 'reg_lambda': 0.020584504089957503}. Best is trial 0 with value: 0.41395181451450397.
[I 2025-05-16 12:12:22,904] Trial 1 finished with value: 0.42365702178942893 and parameters: {'booster': 'gbtree', 'learning_rate': 0.020589728197687916, 'n_estimators': 172, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 0.2623782158161189, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'reg_alpha': 0.6118528986038505, 'reg_lambda': 0.13949386925710322}. Best is trial 1 with value: 0.42365702178942893.
[I 2025-05-16 12:12:2


Fold 1 Metrics:
Val Accuracy: 0.5303
Val F1: 0.3929


[I 2025-05-16 12:17:30,082] Trial 0 finished with value: 0.4107282549598641 and parameters: {'booster': 'dart', 'learning_rate': 0.1205712628744377, 'n_estimators': 340, 'max_depth': 4, 'min_child_weight': 2, 'gamma': 0.02904180608409973, 'subsample': 0.9330880728874675, 'colsample_bytree': 0.8005575058716043, 'reg_alpha': 0.7080725807153198, 'reg_lambda': 0.020584504089957503}. Best is trial 0 with value: 0.4107282549598641.
[I 2025-05-16 12:17:31,832] Trial 1 finished with value: 0.42178240880851514 and parameters: {'booster': 'gbtree', 'learning_rate': 0.020589728197687916, 'n_estimators': 172, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 0.2623782158161189, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'reg_alpha': 0.6118528986038505, 'reg_lambda': 0.13949386925710322}. Best is trial 1 with value: 0.42178240880851514.
[I 2025-05-16 12:17:35,111] Trial 2 finished with value: 0.42568230253459266 and parameters: {'booster': 'dart', 'learning_rate': 0.04717


Fold 2 Metrics:
Val Accuracy: 0.4596


[I 2025-05-16 12:29:53,271] A new study created in memory with name: no-name-2c306407-cd25-4168-9d3b-bb38ecca2d7f


...


Fold 4 Metrics:
Val Accuracy: 0.5095


[I 2025-05-16 13:01:49,510] A new study created in memory with name: no-name-0847c807-2ce2-4d98-9f18-5b455c3f1f50


Val F1: 0.3714


[I 2025-05-16 13:01:51,313] Trial 0 finished with value: 0.40433844392764823 and parameters: {'booster': 'dart', 'learning_rate': 0.1205712628744377, 'n_estimators': 340, 'max_depth': 4, 'min_child_weight': 2, 'gamma': 0.02904180608409973, 'subsample': 0.9330880728874675, 'colsample_bytree': 0.8005575058716043, 'reg_alpha': 0.7080725807153198, 'reg_lambda': 0.020584504089957503}. Best is trial 0 with value: 0.40433844392764823.
[I 2025-05-16 13:01:51,884] Trial 1 finished with value: 0.40196924484854163 and parameters: {'booster': 'gbtree', 'learning_rate': 0.020589728197687916, 'n_estimators': 172, 'max_depth': 4, 'min_child_weight': 4, 'gamma': 0.2623782158161189, 'subsample': 0.7159725093210578, 'colsample_bytree': 0.645614570099021, 'reg_alpha': 0.6118528986038505, 'reg_lambda': 0.13949386925710322}. Best is trial 0 with value: 0.40433844392764823.
[I 2025-05-16 13:01:55,927] Trial 2 finished with value: 0.4207232376073374 and parameters: {'booster': 'dart', 'learning_rate': 0.0471


Fold 5 Metrics:
Val Accuracy: 0.4703
Val F1: 0.3390


In [5]:
predictions = predict_fixtures('fixtures.csv')
predictions['Prob_Diff'] = abs(predictions['Home_Win_Prob'] - predictions['Away_Win_Prob'])
final_predictions = predictions.sort_values('Prob_Diff', ascending=False)

In [6]:
final_predictions

,Home_Team,Away_Team,Home_Win_Prob,Draw_Prob,Away_Win_Prob,Predicted_Home_xG,Predicted_Away_xG,Prob_Diff
5,Bayern Munich,Mainz 05,0.589694,0.240479,0.169826,2.372076,0.803845,0.419868
1,Newcastle United,Ipswich Town,0.561407,0.245159,0.193435,2.125648,0.713358,0.367972
10,Strasbourg,Saint-Étienne,0.549056,0.268986,0.181958,2.203870,1.020554,0.367098
12,Lyon,Rennes,0.546602,0.257791,0.195607,1.848363,1.186009,0.350995
9,Eint Frankfurt,RB Leipzig,0.543049,0.249832,0.207119,1.997337,0.841382,0.335930
3,Wolves,Leicester City,0.526932,0.269052,0.204017,1.489019,0.961060,0.322915
4,Leverkusen,Augsburg,0.530310,0.257508,0.212182,2.003819,0.717562,0.318129
21,Barcelona,Real Madrid,0.513170,0.261797,0.225034,3.156034,1.579366,0.288136
0,Brighton,West Ham,0.481207,0.292075,0.226718,1.468450,1.220235,0.254489
13,Blackburn,Watford,0.489618,0.268257,0.242125,1.343651,0.995549,0.247492
